# Три способа прогнозировать объём сети
     
Все варианты используют доли маршрутов одного CatBoost depth 4 на исходных OOF. Контроль сохраняет объём Session6. Ridge прогнозирует отношение к Session6 с общими признаками утра, поправками горизонтов и расхождениями баз. Третий вариант прогнозирует объём независимо от Session6: отдельный Ridge корректирует сезонный сетевой профиль. Прогноз смешивается с Session6 в пропорциях 25%, 50% и 100%.

На неделях 4–10 и 11–17 октября выбираем настройки двух семейств и общий вариант по среднему WAPE + |Bias|. Train содержит только цели до соответствующей недели. На test сравниваем контроль и по одному заранее выбранному варианту каждого семейства. Доступные до 10:30 наблюдения используются как признаки. Метрика объёма считается по парам день × горизонт, поэтому ошибки не гасятся между горизонтами.

Источники — исходный blend_best_output и исходная размеченная история. Оценка использует архивные прогнозы баз, поэтому её числа не являются оценкой эксперимента trim25 или сабмита на Kaggle. Локальный test уже изучался. Оценка test получена до полного переобучения. Ноутбук запускается на CPU и не создаёт конкурсные CSV.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display


HORIZONS = tuple(range(1, 9))
STEP = pd.Timedelta(minutes=30)
OOF_START = pd.Timestamp('2025-09-20')
LOCAL_TEST_START = pd.Timestamp('2025-10-18')
LOCAL_TEST_END = pd.Timestamp('2025-11-01')
OOF_WINDOWS = list(zip(pd.date_range(OOF_START, periods=4, freq='7D'),
                       pd.date_range(OOF_START + pd.Timedelta(days=7), periods=4, freq='7D')))
KEYS = ['cutoff', 'horizon', 'route_id']
GROUP_KEYS = KEYS[:2]
BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
SEED = 42
THREADS = 4


SESSION_FILES = ('comparison_meta.json', 'train_solo_track.parquet', 'test_solo_track.parquet',
                 'validation_predictions.parquet')
XGB_STAGES = ('oof_week_0', 'oof_week_1', 'oof_week_2', 'calibration_select', 'train_val')
XGB_FILES = ('manifest.json', 'protocol.json', 'evaluation_complete.json',
             *(f'{stage}_h{h}_xgb_k8.parquet' for stage in XGB_STAGES for h in HORIZONS))


def find_dataset(label, path, marker, required):
    if path is None:
        root = Path('/kaggle/input')
        candidates = sorted({file.parent for file in root.rglob(marker)})
        complete = [folder for folder in candidates if all((folder / name).is_file() for name in required)]
        if len(complete) == 1:
            return complete[0]
        if not complete and len(candidates) == 1:
            path = candidates[0]
        else:
            raise FileNotFoundError(f'{label}: нужен один полный набор. Укажите путь к папке в ячейке настроек. '
                                    f'Найденные полные наборы: {complete}; папки с {marker}: {candidates}')
    path = Path(path)
    missing = [name for name in required if not (path / name).is_file()]
    if missing:
        raise FileNotFoundError(f'{label}: в {path} отсутствуют файлы {missing}')
    return path


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, data):
    temp = path.with_suffix('.partial')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


def save_parquet(path, data):
    temp = path.with_suffix('.partial')
    data.to_parquet(temp, index=False)
    temp.replace(path)


def initialize(session_path, xgb_path):
    global ARTIFACTS, TIMES, ROUTES, VALUES, SESSION_META, SESSION_DATA, XGB_DATA
    SESSION_DATA = find_dataset('Session6', session_path, 'comparison_meta.json', SESSION_FILES)
    XGB_DATA = find_dataset('XGBoost rich k8', xgb_path, 'oof_week_0_h1_xgb_k8.parquet', XGB_FILES)
    train_path = SESSION_DATA / 'train_solo_track.parquet'
    SESSION_META = json.loads((SESSION_DATA / 'comparison_meta.json').read_text())
    xgb_manifest = json.loads((XGB_DATA / 'manifest.json').read_text())
    XGB_PROTOCOL = json.loads((XGB_DATA / 'protocol.json').read_text())
    xgb_evaluation = json.loads((XGB_DATA / 'evaluation_complete.json').read_text())
    contract = dict(code=NOTEBOOK_HASH, seed=SEED,
                    session_sha256={name: digest_file(SESSION_DATA / name) for name in SESSION_FILES},
                    xgb_sha256={name: digest_file(XGB_DATA / name) for name in XGB_FILES})
    if (xgb_manifest['train_hash'] != contract['session_sha256']['train_solo_track.parquet'] or
            xgb_manifest['test_hash'] != contract['session_sha256']['test_solo_track.parquet']):
        raise ValueError('Датасеты XGBoost rich и Session6 используют разные входные файлы')
    if (pd.Timestamp(XGB_PROTOCOL['val_start']) != OOF_START or
            pd.Timestamp(XGB_PROTOCOL['calibration_split']) != OOF_WINDOWS[-1][0] or
            pd.Timestamp(XGB_PROTOCOL['test_start']) != LOCAL_TEST_START or
            pd.Timestamp(xgb_evaluation['train_val_model_target_end_exclusive']) != LOCAL_TEST_START or
            XGB_PROTOCOL['model_clusters'] != [8] or XGB_PROTOCOL['test_used_for_selection'] or
            not xgb_evaluation['test_evaluated_before_full_refit']):
        raise ValueError('Границы OOF или протокол архива XGB rich отличаются')
    run_id = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:12]
    ARTIFACTS = Path('/kaggle/working') / f'best_blend_{run_id}'
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    write_json(ARTIFACTS / 'inputs.json', dict(contract, session_path=str(SESSION_DATA), xgb_path=str(XGB_DATA)))
    print('Session6:', SESSION_DATA)
    print('XGBoost rich k8:', XGB_DATA)
    history = pd.read_parquet(train_path, columns=['timestamp', 'route_id', TARGET, *STATUSES])
    history = history.sort_values(['timestamp', 'route_id'])
    TIMES = pd.DatetimeIndex(history.timestamp.unique())
    ROUTES = np.sort(history.route_id.unique())
    if (len(history) != len(TIMES) * len(ROUTES) or history.isna().any().any() or
            history.duplicated(['timestamp', 'route_id']).any() or
            not TIMES.equals(pd.date_range(TIMES[0], TIMES[-1], freq=STEP))):
        raise ValueError('Требуется полная получасовая панель')
    VALUES = {name: history[name].to_numpy(dtype=np.float32).reshape(len(TIMES), len(ROUTES))
              for name in (TARGET, *STATUSES)}
    if not all(np.isfinite(array).all() for array in VALUES.values()) or np.any(VALUES[TARGET] < 0):
        raise ValueError('Некорректные наблюдения исходной панели')
    display(pd.DataFrame([dict(split=f'oof_{i}', start=start, end_exclusive=stop)
                          for i, (start, stop) in enumerate(OOF_WINDOWS, 1)] +
                         [dict(split='test', start=LOCAL_TEST_START, end_exclusive=LOCAL_TEST_END)]))


def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))


def fit_report(model, boundary, horizon, group, split, train_y, train_pred, query_y, query_pred):
    report = pd.DataFrame([
        dict(model=model, fit_before=boundary, horizon=horizon, group=group, period=period, **metric(y, pred))
        for period, y, pred in [('train', train_y, train_pred), (split, query_y, query_pred)]
    ])
    display(report)
    path = ARTIFACTS / f'fits_{model}_{boundary:%Y%m%d}_h{horizon}_{group}.csv'
    report.to_csv(path, index=False)


def read_xgb(stage):
    if stage not in XGB_STAGES:
        raise ValueError(f'Неизвестный этап XGBoost rich: {stage}')
    frames = [pd.read_parquet(XGB_DATA / f'{stage}_h{h}_xgb_k8.parquet',
                              columns=['cutoff', 'timestamp', 'route_id', 'h', 'y', 'y_pred']) for h in HORIZONS]
    result = pd.concat(frames, ignore_index=True).rename(columns={'h': 'horizon', 'y_pred': 'xgb_rich'})
    if not result.timestamp.eq(result.cutoff + result.horizon * STEP).all():
        raise ValueError('Неверный timestamp в XGB rich')
    return result


def read_session6_test():
    frame = pd.read_parquet(SESSION_DATA / 'validation_predictions.parquet',
                            columns=KEYS + ['y', 'session_prediction'])
    return frame.rename(columns={'session_prediction': 'session6'})


def context(start, stop):
    origins = np.flatnonzero((TIMES >= start) & (TIMES < stop) & (TIMES.hour == 10) & (TIMES.minute == 30))
    expected_days = (stop - start).days
    if len(origins) != expected_days or origins[-1] + max(HORIZONS) >= len(TIMES):
        raise ValueError('Не хватает дней или целевых горизонтов')
    y = VALUES[TARGET]
    windows = np.arange(-47, 1)
    level24 = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64) for pos in origins])
    seasonal_origins = origins[:, None] - 48 * np.arange(1, 29)
    if seasonal_origins.min() + windows.min() < 0:
        raise ValueError('Не хватает наблюдений для profile28')
    seasonal_level = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64)
                               for pos in seasonal_origins.ravel()]).reshape(*seasonal_origins.shape, len(ROUTES))
    ratio = level24[:, None, :] / np.maximum(seasonal_level, 1)
    route_count = len(ROUTES)
    common = dict(cutoff=TIMES[origins].repeat(route_count), route_id=np.tile(ROUTES, len(origins)),
                  level24=level24.ravel(), current=y[origins].ravel(),
                  delta1h=(y[origins] - y[origins - 2]).ravel(),
                  delta2h=(y[origins] - y[origins - 4]).ravel(),
                  network_current=np.repeat(y[origins].sum(axis=1, dtype=np.float64), route_count))
    return pd.concat([pd.DataFrame(dict(common, horizon=h, y=y[origins + h].ravel(),
                                        profile28=(y[seasonal_origins + h] * ratio).mean(axis=1).ravel()))
                      for h in HORIZONS], ignore_index=True).sort_values(KEYS).reset_index(drop=True)


def join_base(frame, incoming, column):
    relevant = incoming[KEYS + ['y', column]]
    merged = frame.merge(relevant, on=KEYS, how='left', validate='one_to_one', suffixes=('', '_incoming'))
    if len(relevant) != len(frame) or not np.array_equal(merged.y, merged.y_incoming):
        raise ValueError(f'Не совпали ключи или факты для {column}')
    if not np.isfinite(merged[column]).all() or merged[column].lt(0).any():
        raise ValueError(f'Некорректные прогнозы {column}')
    return merged.drop(columns='y_incoming')


def assemble(start, stop, session, rf, cat, xgb, xgb_clean):
    frame = context(start, stop)
    for source, name in [(session, 'session6'), (rf, 'rf'), (cat, 'cat_rich'),
                         (xgb, 'xgb_rich'), (xgb_clean, 'xgb_clean')]:
        part = source.loc[source.cutoff.ge(start) & source.cutoff.lt(stop)]
        frame = join_base(frame, part, name)
    return frame


def archive_report():
    path = ARTIFACTS.with_suffix('.zip')
    temp = path.with_suffix('.partial')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as archive:
        for file in sorted(ARTIFACTS.rglob('*')):
            if file.is_file() and not file.name.endswith('.partial'):
                archive.write(file, file.relative_to(ARTIFACTS.parent))
    temp.replace(path)
    print('Архив результатов', path)


In [ ]:
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-05):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (0.2, 0.05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-09:
                        best = loss
                        choice = (source, destination, amount, trial)
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(log_level=np.log1p(level), current=frame.current.to_numpy() / level, delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level, log_network=np.log1p(frame.network_current.to_numpy()), horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(), shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return (context.clip(-50, 50).to_numpy(dtype=np.float32), level)

def fit_shape(name, frame, matrix, joint):
    if name == 'session6':
        return None
    if name == 'equal':
        return np.full(len(BASES), 1 / len(BASES))
    if name == 'joint':
        return joint
    if name == 'joint_without_xgb_clean':
        mask = np.array([base != 'xgb_clean' for base in BASES])
        weights = np.zeros(len(BASES))
        weights[mask] = fit_weights(matrix[:, mask], frame.y)
        return weights
    if name.startswith('pair_'):
        index = BASES.index(name.removeprefix('pair_'))
        weights = np.zeros(len(BASES))
        weights[[0, index]] = fit_weights(matrix[:, [0, index]], frame.y)
        return weights
    if name == 'by_horizon':
        weights = []
        for h in HORIZONS:
            mask = frame.horizon.eq(h).to_numpy()
            weights.append(0.5 * joint + 0.5 * fit_weights(matrix[mask], frame.y.to_numpy()[mask]))
        return np.stack(weights)
    baseline = matrix @ joint
    features, level = meta_features(frame, matrix, baseline)
    residual = (frame.y.to_numpy(dtype=np.float64) - baseline) / level
    sample_weight = level / level.mean()
    if name.startswith('gating_'):
        return fit_gating(features, matrix, frame.y, level, joint, int(name.split('_')[-1]))
    if name.startswith('ridge_'):
        prep = StandardScaler().fit(features)
        model = Ridge(alpha=float(name.split('_')[-1]))
        model.fit(prep.transform(features), residual, sample_weight=sample_weight)
        return (prep, model)
    depth = int(name[-1])
    if name.startswith('cat_'):
        model = CatBoostRegressor(depth=depth, iterations=220, loss_function='MAE', learning_rate=0.035, l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False, thread_count=THREADS)
    else:
        model = XGBRegressor(max_depth=depth, n_estimators=220, objective='reg:absoluteerror', tree_method='hist', learning_rate=0.035, reg_lambda=20, min_child_weight=30, random_state=SEED, n_jobs=THREADS)
    model.fit(features, residual, sample_weight=sample_weight)
    return (None, model)

def predict_shape(name, fitted, frame, matrix, joint):
    if name == 'session6':
        return matrix[:, 0]
    if isinstance(fitted, np.ndarray):
        if fitted.ndim == 1:
            return matrix @ fitted
        return np.sum(matrix * fitted[frame.horizon.to_numpy() - 1], axis=1)
    baseline = matrix @ joint
    prep, model = fitted
    features, level = meta_features(frame, matrix, baseline)
    if name.startswith('gating_'):
        weights = gate_weights(prep, model, features)
        return with_volume(frame, np.sum(matrix * weights, axis=1))
    encoded = features if prep is None else prep.transform(features)
    correction = np.clip(model.predict(encoded), -0.5, 0.5) * level
    return with_volume(frame, baseline + correction)


In [ ]:
from sklearn.pipeline import make_pipeline


def load_network(path):
    history = pd.read_parquet(path, columns=['timestamp', TARGET, *STATUSES])
    if history.isna().any().any():
        raise ValueError('Пропуски в истории сети')
    network = history.groupby('timestamp', sort=True)[[TARGET, *STATUSES]].sum()
    if not network.index.equals(pd.date_range(network.index[0], network.index[-1], freq=STEP)):
        raise ValueError('Неполная получасовая история')
    if not np.isfinite(network.to_numpy()).all() or network.lt(0).any().any():
        raise ValueError('Некорректные объёмы')
    return network


def network_features(network):
    times = network.index
    values = network.to_numpy(dtype=np.float64)
    positions = np.flatnonzero((times.hour == 10) & (times.minute == 30) &
                               (np.arange(len(times)) >= 29 * 48))
    means = pd.Series(values[:, 0]).rolling(48).mean().to_numpy()
    days = positions[:, None] - 48 * np.arange(1, 29)
    common = {}
    for i, name in enumerate(network.columns):
        current = values[positions, i]
        common[f'{name}_level'] = np.log1p(current)
        for lag in [2, 4, 8]:
            common[f'{name}_change_{lag}'] = np.log1p(current) - np.log1p(values[positions-lag, i])
        common[f'{name}_seasonal_deviation'] = np.log1p(current) - np.log1p(np.median(values[days, i], axis=1))
    frames = []
    for h in HORIZONS:
        seasonal = np.median(values[days+h, 0], axis=1)
        scale = means[positions] / np.maximum(np.median(means[days], axis=1), 1.)
        timestamp = times[positions] + h * STEP
        target = np.full(len(positions), np.nan)
        known = positions+h < len(times)
        target[known] = values[positions[known]+h, 0]
        frames.append(pd.DataFrame(dict(common, cutoff=times[positions], timestamp=timestamp, horizon=h,
                                        network_profile=np.maximum(seasonal*scale, 1.), actual_volume=target)))
    return pd.concat(frames, ignore_index=True)


def aggregate(frame, network):
    values = frame[BASES + ['y']].astype(np.float64)
    totals = frame[GROUP_KEYS].join(values).groupby(GROUP_KEYS, as_index=False).sum()
    table = totals.merge(network, on=GROUP_KEYS, validate='one_to_one')
    if (len(table) != len(totals) or not np.allclose(table.y, table.actual_volume, rtol=1e-7, atol=1)
            or table.session6.le(0).any()):
        raise ValueError('Объёмы архивных прогнозов и исходные факты не согласованы')
    return table


def design(table, with_anchor):
    cols = [c for c in table if c.endswith('_level') or '_change_' in c or c.endswith('_seasonal_deviation')]
    features = table[cols].copy()
    features['log_profile'] = np.log(table.network_profile)
    for h in HORIZONS[1:]:
        features[f'horizon_{h}'] = table.horizon.eq(h).astype(float)
    day = table.cutoff.dt.dayofweek
    features['weekday_sin'] = np.sin(2*np.pi*day/7)
    features['weekday_cos'] = np.cos(2*np.pi*day/7)
    features['horizon_trend'] = (table.horizon - 4.5) * features[f'{TARGET}_change_4']
    if with_anchor:
        anchor = table.session6.to_numpy()
        features['log_anchor'] = np.log(anchor)
        for name in BASES[1:]:
            features[f'{name}_vs_anchor'] = np.log((table[name]+1)/(anchor+1))
    return features


def fit_predict_volumes(train, query, network, boundary):
    if not train.timestamp.lt(boundary).all():
        raise ValueError('Обучающие цели пересекли границу')
    ratios = np.log((train.y+1)/(train.session6+1))
    model = make_pipeline(StandardScaler(), Ridge(alpha=100.))
    model.fit(design(train, True), ratios)
    daily_log_ratio = model.predict(design(query, True))
    earlier = network.loc[network.timestamp.lt(boundary) & network.actual_volume.notna()]
    if earlier.cutoff.nunique() < 14:
        raise ValueError('Мало утренней истории')
    direct = make_pipeline(StandardScaler(), Ridge(alpha=100.))
    direct.fit(design(earlier, False), np.log((earlier.actual_volume+1)/(earlier.network_profile+1)))
    independent = query.network_profile.to_numpy() * np.exp(direct.predict(design(query, False)))
    anchor = query.session6.to_numpy()
    result = {'session6': anchor}
    for strength in (.25, .5, 1.):
        result[f'coefficient_{strength:g}'] = anchor * np.clip(np.exp(strength*daily_log_ratio), .95, 1.05)
    for weight in (.25, .5, 1.):
        result[f'direct_{weight:g}'] = (1-weight)*anchor + weight*independent
    return result


def score_variants(train, query, network, boundary, period, only=None):
    matrix = shape_matrix(train)
    joint = fit_weights(matrix, train.y)
    fitted = fit_shape('cat_d4', train, matrix, joint)
    rows = []
    output = {}
    for label, frame in [('train_oof', train), (period, query)]:
        shape = predict_shape('cat_d4', fitted, frame, shape_matrix(frame), joint)
        train_volume = aggregate(train, network)
        query_volume = aggregate(frame, network)
        variants = fit_predict_volumes(train_volume, query_volume, network, boundary)
        totals = frame[GROUP_KEYS].assign(shape=shape).groupby(GROUP_KEYS).shape.transform('sum').to_numpy()
        if np.any(totals <= 0):
            raise ValueError('Нулевой объём формы')
        shares = shape / totals
        for name, volume in variants.items():
            if only is not None and name not in only:
                continue
            mapped = frame[GROUP_KEYS].merge(query_volume[GROUP_KEYS].assign(volume=volume), on=GROUP_KEYS,
                                             validate='many_to_one', how='left').volume.to_numpy()
            prediction = shares*mapped
            stats = metric(frame.y, prediction)
            volume_error = volume - query_volume.y.to_numpy()
            volume_wape = np.abs(volume_error).sum()/query_volume.y.sum()
            rows.append(dict(period=label, variant=name, volume_WAPE=volume_wape, **stats))
            if label == period:
                output[name] = prediction
    return pd.DataFrame(rows), output


In [ ]:
import os

INPUT_ROOT = Path(os.environ.get('BLEND_INPUT', 'blend_inputs'))
EVALUATION_DIR = INPUT_ROOT / 'blend_best_output'
HISTORY_PATH = INPUT_ROOT / 'session6/train_solo_track.parquet'
ARTIFACTS = Path(os.environ.get('VOLUME_OUTPUT', 'output_volume_experiment'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
contract = json.loads((EVALUATION_DIR / 'inputs.json').read_text())
if digest_file(HISTORY_PATH) != contract['session_sha256'][HISTORY_PATH.name]:
    raise ValueError('История отличается от исходного запуска')
network = network_features(load_network(HISTORY_PATH))
columns = KEYS + ['y', *BASES, 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
oof = pd.read_parquet(EVALUATION_DIR / 'base_oof.parquet', columns=columns).sort_values(KEYS).reset_index(drop=True)
targets = oof.cutoff + oof.horizon * STEP
if oof.duplicated(KEYS).any() or not targets.lt(LOCAL_TEST_START).all():
    raise ValueError('Некорректные ключи или границы OOF')


In [ ]:
blocks = []
for start, stop in OOF_WINDOWS[2:]:
    train = oof.loc[targets.lt(start)]
    val = oof.loc[oof.cutoff.ge(start) & oof.cutoff.lt(stop)]
    table, _ = score_variants(train, val, network, start, 'val')
    table['week'] = str(start.date())
    display(table)
    blocks.append(table)
validation = pd.concat(blocks, ignore_index=True)
validation.to_csv(ARTIFACTS / 'validation_metrics.csv', index=False)
ranking = validation.loc[validation.period.eq('val')].groupby('variant', as_index=False).agg(
    mean_score=('score', 'mean'), worst_score=('score', 'max'), mean_volume_WAPE=('volume_WAPE', 'mean'), weeks=('score', 'count'))
if not ranking.weeks.eq(2).all():
    raise ValueError('Неполная validation')
ranking = ranking.sort_values(['mean_score', 'worst_score', 'variant'])
selected = ['session6', *[ranking.loc[ranking.variant.str.startswith(prefix), 'variant'].iloc[0]
                         for prefix in ['coefficient_', 'direct_']]]
frozen = dict(variants=selected, winner=ranking.variant.iloc[0], selected_before_test=True)
write_json(ARTIFACTS / 'frozen_selection.json', frozen)
ranking.to_csv(ARTIFACTS / 'validation_ranking.csv', index=False)
display(ranking)


In [ ]:
frozen = json.loads((ARTIFACTS / 'frozen_selection.json').read_text())
test = pd.read_parquet(EVALUATION_DIR / 'test_predictions.parquet', columns=columns).sort_values(KEYS).reset_index(drop=True)
if (test.duplicated(KEYS).any() or not test.cutoff.ge(LOCAL_TEST_START).all() or
        not (test.cutoff + test.horizon*STEP).lt(LOCAL_TEST_END).all()):
    raise ValueError('Некорректные ключи или границы test')
report, predictions = score_variants(oof, test, network, LOCAL_TEST_START, 'test', frozen['variants'])
report['winner_on_val'] = report.variant.eq(frozen['winner'])
report.to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
display(report)
rows = test[KEYS + ['y']].assign(**predictions)
save_parquet(ARTIFACTS / 'test_predictions.parquet', rows)
volumes = rows.groupby(GROUP_KEYS, as_index=False)[['y', *predictions]].sum()
volumes.to_csv(ARTIFACTS / 'test_volume_by_day_horizon.csv', index=False)
print('Оценка test получена до полного переобучения. Конкурсный прогноз не создавался.')
